In [48]:
import io
import os
import zipfile
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
from google.cloud import storage
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from google.cloud import storage
import os
# Force legacy Keras configuration before any TF imports
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
from tf_keras import layers, models, callbacks
import tensorflow_hub as hub


In [46]:
from google.colab import auth
auth.authenticate_user()

print("Successfully authenticated with Google Cloud!")

Successfully authenticated with Google Cloud!


### Preprocessing

In [49]:
# --- 1. INITIALIZE GCS IN-MEMORY BUFFER ---

BUCKET_NAME = 'urban_sound'
ARCHIVE_NAME = 'ESC50.zip'

print(f"Streaming data directly from bucket '{BUCKET_NAME}' into RAM...")

storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)
blob = bucket.blob(ARCHIVE_NAME)

# Download archive directly into RAM
gcs_data_buffer = io.BytesIO()
blob.download_to_file(gcs_data_buffer)
gcs_data_buffer.seek(0)

# Open ZIP archive from RAM
archive = zipfile.ZipFile(gcs_data_buffer)


# --- 2. EXTRACT METADATA IN-MEMORY ---

print("Extracting metadata table...")

metadata_file_name = "ESC-50-master/meta/esc50.csv"

with archive.open(metadata_file_name) as csv_file:
    metadata = pd.read_csv(csv_file)


# Build paths to audio files inside the ZIP
metadata["filepath"] = metadata["filename"].apply(
    lambda x: f"ESC-50-master/audio/{x}"
)

X_paths = metadata["filepath"].values
y = metadata["target"].values


# --- 3. TRAIN / TEST SPLIT ---

X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    X_paths,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# --- 4. LOAD YAMNET ---

print("Loading YAMNet...")

yamnet_model = hub.load(
    "https://tfhub.dev/google/yamnet/1"
)


# --- 5. CREATE QUICK LOOKUPS ---

train_set = set(X_train_paths)
test_set = set(X_test_paths)


# --- 6. CACHE AUDIO BYTES IN RAM ---

audio_bytes_cache = {}

print("Caching target audio bytes from archive...")

for member_name in tqdm(archive.namelist()):

    if member_name.endswith(".wav"):

        if member_name in train_set or member_name in test_set:

            audio_bytes_cache[member_name] = archive.read(member_name)


# Close ZIP archive
archive.close()


# --- 7. AUDIO LOADER ---

def load_audio(file_path):
    """
    Load WAV file as mono waveform at 16kHz
    directly from the in-memory cache.
    """

    raw_bytes = audio_bytes_cache[file_path]

    waveform, sr = librosa.load(
        io.BytesIO(raw_bytes),
        sr=16000,
        mono=True
    )

    return waveform.astype(np.float32)


# --- 8. EXTRACT YAMNET EMBEDDING ---

def extract_embedding(file_path):
    """
    Extract a single 1024-dimensional YAMNet embedding
    from an audio file.
    """

    waveform = load_audio(file_path)

    scores, embeddings, spectrogram = yamnet_model(
        waveform
    )

    feature_vector = tf.reduce_mean(
        embeddings,
        axis=0
    )

    return feature_vector.numpy()


# --- 9. EXTRACT TRAINING EMBEDDINGS ---

print("\nExtracting training embeddings...")

X_train = np.array([
    extract_embedding(path)
    for path in tqdm(X_train_paths)
])


# --- 10. EXTRACT TEST EMBEDDINGS ---

print("\nExtracting test embeddings...")

X_test = np.array([
    extract_embedding(path)
    for path in tqdm(X_test_paths)
])


# --- 11. FINAL OUTPUT ---

print("\nAll processing completed entirely in-memory!")

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"Number of classes: {len(np.unique(y))}")


Streaming data directly from bucket 'urban_sound' into RAM...
Extracting metadata table...
Loading YAMNet...
Caching target audio bytes from archive...


100%|██████████| 2017/2017 [00:05<00:00, 359.47it/s]



Extracting training embeddings...


100%|██████████| 1600/1600 [00:14<00:00, 111.73it/s]



Extracting test embeddings...


100%|██████████| 400/400 [00:03<00:00, 120.40it/s]


All processing completed entirely in-memory!
X_train shape: (1600, 1024)
X_test shape: (400, 1024)
y_train shape: (1600,)
y_test shape: (400,)
Number of classes: 50


### Model

In [37]:
# Model in use
#audio_input = layers.Input(shape=(1024,), dtype=tf.float32, name="audio_waveform")
#x = layers.Dense(256, activation='relu')(audio_input)
#x = layers.Dropout(0.2)(x)
#x = layers.Dense(128, activation='relu')(x)
#x = layers.Dropout(0.2)(x)
#x = layers.GlobalAveragePooling1D()(x)
#outputs = layers.Dense(50, activation='softmax')(x)

In [50]:
#Test new model
audio_input = layers.Input(shape=(1024,), dtype=tf.float32, name="audio_waveform")
x = layers.Dense(64, activation='relu')(audio_input)

outputs = layers.Dense(50, activation='softmax')(x)

In [51]:
# 3. Instantiate and compile model
import keras

model = models.Model(inputs=audio_input, outputs=outputs)
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [52]:
# 4. Set up early stopping callback
es = callbacks.EarlyStopping(patience=10, restore_best_weights=True)

# 5. Fit the model using your exact processing variables
model.fit(
    X_train,
    y_train,
    batch_size=32,
    epochs=1000,
    verbose=1,
    callbacks=[es],
    validation_split=0.2
)


Epoch 1/1000
40/40 [==============================] - 1s 10ms/step - loss: 3.5114 - accuracy: 0.1789 - val_loss: 2.9217 - val_accuracy: 0.4187
Epoch 2/1000
40/40 [==============================] - 0s 5ms/step - loss: 2.4359 - accuracy: 0.5242 - val_loss: 2.0093 - val_accuracy: 0.6000
Epoch 3/1000
40/40 [==============================] - 0s 6ms/step - loss: 1.7139 - accuracy: 0.6562 - val_loss: 1.5515 - val_accuracy: 0.6594
Epoch 4/1000
40/40 [==============================] - 0s 7ms/step - loss: 1.3305 - accuracy: 0.7148 - val_loss: 1.3331 - val_accuracy: 0.6687
Epoch 5/1000
40/40 [==============================] - 0s 4ms/step - loss: 1.0858 - accuracy: 0.7516 - val_loss: 1.2070 - val_accuracy: 0.6875
Epoch 6/1000
40/40 [==============================] - 0s 4ms/step - loss: 0.9761 - accuracy: 0.7750 - val_loss: 1.0409 - val_accuracy: 0.7531
Epoch 7/1000
40/40 [==============================] - 0s 4ms/step - loss: 0.8643 - accuracy: 0.7914 - val_loss: 0.9883 - val_accuracy: 0.7344
Epoch

In [53]:
# 6. Evaluate on test set
model.evaluate(X_test, y_test)

13/13 [==============================] - 0s 3ms/step - loss: 0.5223 - accuracy: 0.8475


[0.5222954750061035, 0.8475000262260437]

In [54]:
model.save('dense_50.keras')

In [55]:

def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    # Initializes a client
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)

    # Uploads the local file
    blob.upload_from_filename(source_file_name)
    print(f"File {source_file_name} uploaded to {destination_blob_name}.")

# Usage
upload_to_gcs('spectra_model', 'dense_50.keras', 'models/dense_50.keras')


File dense_50.keras uploaded to models/dense_50.keras.
